# PNS-Guided ICL — Inference Notebook (10 Test Questions)\n\nFive ICL methods on **Qwen3-8B** — prompt-only, no fine-tuning, no adapter.\n\n| Method | Description |\n|---|---|\n| **Standard CoT** | Verbose step-by-step reasoning |\n| **Fast Solve** | Brief efficient solution, skip obvious sub-steps |\n| **Reduction** | Moderately compressed; drops trivially redundant steps |\n| **CoD** | Chain of Draft — ultra-brief key steps only |\n| **Ours ICL** | PNS-optimized traces as few-shot demos (causally necessary steps only) |\n\n**Runtime → Change runtime type → T4 GPU** (free tier is sufficient).  \nQwen3-8B in 4-bit NF4 uses ~5.5 GB VRAM — fits on T4 with headroom.

## 1 · Install dependencies

In [ ]:
!pip install -q --upgrade transformers accelerate bitsandbytes sentencepiece

## 2 · Imports & quantisation config

In [ ]:
import re, time, json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), "GPU required — Runtime → Change runtime type → T4 GPU"

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
BNB_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

BASE_MODEL = "Qwen/Qwen3-8B"

# Per-dataset token limits — matches paper's run_icl.py
MAX_NEW_TOKENS = {"gsm8k": 1536, "math500": 8192}

print(f"GPU: {torch.cuda.get_device_name(0)} | dtype: {COMPUTE_DTYPE}")
print(f"Model: {BASE_MODEL}")

## 3 · Load Qwen3-8B base model (4-bit NF4, no adapter)\n\nICL is **prompt-only** — no fine-tuning, no LoRA adapter.\n\n> **VRAM**: ~5.5 GB in 4-bit NF4. Fits on T4 (15 GB) with plenty of headroom. No A100 needed.

In [ ]:
print(f"Loading {BASE_MODEL} in 4-bit NF4 ...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=BNB_CONFIG,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
model.eval()
print(f"VRAM used: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print("Ready.")

## 4 · Answer extraction helpers

In [ ]:
def _extract_boxed(text: str) -> str:
    results, start = [], 0
    while True:
        idx = text.find(r"\boxed{", start)
        if idx == -1:
            break
        depth = 0
        for i in range(idx + 7, len(text)):
            if text[i] == "{":   depth += 1
            elif text[i] == "}":
                if depth == 0:
                    results.append(text[idx + 7:i])
                    start = i + 1
                    break
                depth -= 1
        else:
            break
    return results[-1].strip() if results else ""


def _normalize(text: str) -> str:
    t = text.strip()
    t = re.sub(r"\\left|\\right|\\,|\\!", "", t)
    t = re.sub(r"\^\\circ|\\circ|\\degree|°", "", t)
    t = re.sub(r"\\text\{([^}]*)\}", r"\1", t)
    t = re.sub(r"\\dfrac", r"\\frac", t)
    t = re.sub(r"\\tfrac", r"\\frac", t)
    t = re.sub(r"\$+", "", t)
    t = re.sub(r"\s+", "", t)
    return t.lower()


def is_correct(extracted: str, ground_truth: str) -> bool:
    if not extracted:
        return False
    if _normalize(extracted) == _normalize(ground_truth):
        return True
    try:
        return float(extracted.replace(",", "").strip()) == \
               float(str(ground_truth).replace(",", "").strip())
    except (ValueError, TypeError):
        return False


def extract_answer(text: str, dataset: str) -> str:
    boxed = _extract_boxed(text)
    if boxed:
        return boxed
    if "gsm" in dataset.lower():
        m = re.search(r"####\s*([\d,.\-]+)", text)
        if m:
            return m.group(1).replace(",", "").strip()
    return ""


def strip_think(text: str) -> str:
    """Remove any residual <think>...</think> block (Qwen3 with thinking disabled
    should not emit one, but guard just in case)."""
    if "</think>" in text:
        return text.split("</think>", 1)[1].strip()
    return text.strip()


def count_steps(text: str) -> int:
    return len([p for p in re.split(r"\n\n+", text) if p.strip()])


print("Helpers loaded.")

## 5 · ICL prompt builders

Inlined from `expt/icl_prompts.py` — five methods × two datasets.

### Data consistency design

All 5 methods use **identical few-shot questions** per dataset. Only the **answer style** changes:

| Method | Few-shot answer style | System instruction |
|---|---|---|
| Standard CoT | Verbose, every sub-step spelled out | "Show every reasoning step clearly" |
| Fast Solve | 1–2 lines, obvious sub-steps skipped | "Brief but complete" |
| Reduction | Key logical steps, trivially redundant steps dropped | "Concisely; omit trivially redundant" |
| CoD | Ultra-brief — key operations only | "Minimum key draft steps" |
| **Ours ICL** | **PNS-style** — every causally necessary step, nothing more | "Sufficient and necessary reasoning" |

**What "Ours ICL" does with PNS chains:**  
The `ours_icl` few-shot answers are written in the style of PNS-optimized (causally pruned) chains: each answer contains only the steps whose removal would cause the solution to fail (PN ≥ α). This teaches the model via demonstration — no fine-tuning, no adapter. The model then generates new answers in that style for unseen test questions.

**Test questions**: The same 10 questions are used for all 5 methods — there is no leakage between few-shot examples and the test set.

In [ ]:
# ── Few-shot examples per method × dataset ───────────────────────────────────
#
# DATA CONSISTENCY: All 5 methods use the SAME 3 base questions per dataset.
# Only the answer style differs — this isolates the effect of reasoning style.
#
# GSM8K base questions: Tom's apples | Sara's notebooks | Train distance
# MATH-500 base questions: Triangle area | Fraction 3/4+1/6 | Linear eq 2x+5=17

# ── GSM8K ─────────────────────────────────────────────────────────────────────

_GSM_STD = [
    {"q": "Tom has 5 apples. He gives 2 to Jane and then buys 3 more. How many apples does Tom have now?",
     "a": "Tom starts with 5 apples.\n\nHe gives 2 to Jane: 5 − 2 = 3 apples remaining.\n\nHe then buys 3 more: 3 + 3 = 6 apples.\n\n#### 6"},
    {"q": "A shop sells notebooks for $3 each. Sara buys 4 notebooks and pays with a $20 bill. How much change does she receive?",
     "a": "Cost of 4 notebooks: 4 × $3 = $12.\n\nSara pays $20, so her change is $20 − $12 = $8.\n\n#### 8"},
    {"q": "A train travels 60 miles per hour for 2.5 hours. How far does it travel?",
     "a": "Distance = speed × time.\n\nDistance = 60 × 2.5 = 150 miles.\n\n#### 150"},
]
_GSM_FAST = [
    {"q": "Tom has 5 apples. He gives 2 to Jane and then buys 3 more. How many apples does Tom have now?",
     "a": "5 − 2 + 3 = 6. #### 6"},
    {"q": "A shop sells notebooks for $3 each. Sara buys 4 notebooks and pays with a $20 bill. How much change does she receive?",
     "a": "4 × 3 = 12. Change = 20 − 12 = 8. #### 8"},
    {"q": "A train travels 60 miles per hour for 2.5 hours. How far does it travel?",
     "a": "60 × 2.5 = 150. #### 150"},
]
_GSM_RED = [
    {"q": "Tom has 5 apples. He gives 2 to Jane and then buys 3 more. How many apples does Tom have now?",
     "a": "After giving 2: 5 − 2 = 3. After buying 3 more: 3 + 3 = 6. #### 6"},
    {"q": "A shop sells notebooks for $3 each. Sara buys 4 notebooks and pays with a $20 bill. How much change does she receive?",
     "a": "Total cost: 4 × 3 = 12. Change: 20 − 12 = 8. #### 8"},
    {"q": "A train travels 60 miles per hour for 2.5 hours. How far does it travel?",
     "a": "Distance = 60 × 2.5 = 150 miles. #### 150"},
]
_GSM_COD = [
    {"q": "Tom has 5 apples. He gives 2 to Jane and then buys 3 more. How many apples does Tom have now?",
     "a": "5−2=3, 3+3=6. #### 6"},
    {"q": "A shop sells notebooks for $3 each. Sara buys 4 notebooks and pays with a $20 bill. How much change does she receive?",
     "a": "4×3=12, 20−12=8. #### 8"},
    {"q": "A train travels 60 miles per hour for 2.5 hours. How far does it travel?",
     "a": "60×2.5=150. #### 150"},
]
# Ours ICL (PNS-style): same 3 questions, answers contain only causally necessary steps
_GSM_PNS = [
    {"question": "Tom has 5 apples. He gives 2 to Jane and then buys 3 more. How many apples does Tom have now?",
     "optimized_chain": "5 − 2 + 3 = 6.\n\n#### 6"},
    {"question": "A shop sells notebooks for $3 each. Sara buys 4 notebooks and pays with a $20 bill. How much change does she receive?",
     "optimized_chain": "4 × $3 = $12; $20 − $12 = $8.\n\n#### 8"},
    {"question": "A train travels 60 miles per hour for 2.5 hours. How far does it travel?",
     "optimized_chain": "60 × 2.5 = 150.\n\n#### 150"},
]

# ── MATH-500 ──────────────────────────────────────────────────────────────────

_MATH_STD = [
    {"q": "Find the area of a triangle with base 6 and height 4.",
     "a": "The formula for the area of a triangle is:\n$$\\text{Area} = \\tfrac{1}{2} \\times \\text{base} \\times \\text{height}$$\n\nSubstituting base = 6 and height = 4:\n$$\\text{Area} = \\tfrac{1}{2} \\times 6 \\times 4 = 12$$\n\nThe area is $\\boxed{12}$."},
    {"q": "Simplify $\\dfrac{3}{4} + \\dfrac{1}{6}$.",
     "a": "Find the least common denominator of 4 and 6, which is 12.\n\nConvert: $\\dfrac{3}{4} = \\dfrac{9}{12}$ and $\\dfrac{1}{6} = \\dfrac{2}{12}$.\n\nAdd: $\\dfrac{9}{12} + \\dfrac{2}{12} = \\dfrac{11}{12}$.\n\nThe answer is $\\boxed{\\dfrac{11}{12}}$."},
    {"q": "What is the value of $x$ if $2x + 5 = 17$?",
     "a": "Subtract 5 from both sides: $2x = 17 - 5 = 12$.\n\nDivide both sides by 2: $x = 6$.\n\n$\\boxed{6}$"},
]
_MATH_FAST = [
    {"q": "Find the area of a triangle with base 6 and height 4.",
     "a": "Area = $\\tfrac{1}{2}(6)(4) = \\boxed{12}$."},
    {"q": "Simplify $\\dfrac{3}{4} + \\dfrac{1}{6}$.",
     "a": "LCD = 12: $\\dfrac{9}{12} + \\dfrac{2}{12} = \\boxed{\\dfrac{11}{12}}$."},
    {"q": "What is the value of $x$ if $2x + 5 = 17$?",
     "a": "$2x = 12 \\Rightarrow x = \\boxed{6}$."},
]
_MATH_RED = [
    {"q": "Find the area of a triangle with base 6 and height 4.",
     "a": "Using Area = $\\tfrac{1}{2} \\cdot b \\cdot h$: $\\tfrac{1}{2}(6)(4) = \\boxed{12}$."},
    {"q": "Simplify $\\dfrac{3}{4} + \\dfrac{1}{6}$.",
     "a": "LCD of 4 and 6 is 12. $\\dfrac{9}{12} + \\dfrac{2}{12} = \\boxed{\\dfrac{11}{12}}$."},
    {"q": "What is the value of $x$ if $2x + 5 = 17$?",
     "a": "$2x = 17 - 5 = 12$, so $x = \\boxed{6}$."},
]
_MATH_COD = [
    {"q": "Find the area of a triangle with base 6 and height 4.",
     "a": "$\\tfrac{1}{2}(6)(4)=\\boxed{12}$"},
    {"q": "Simplify $\\dfrac{3}{4} + \\dfrac{1}{6}$.",
     "a": "LCD=12; $\\tfrac{9+2}{12}=\\boxed{\\tfrac{11}{12}}$"},
    {"q": "What is the value of $x$ if $2x + 5 = 17$?",
     "a": "$2x=12, x=\\boxed{6}$"},
]
# Ours ICL (PNS-style): same 3 questions, answers contain only causally necessary steps
_MATH_PNS = [
    {"question": "Find the area of a triangle with base 6 and height 4.",
     "optimized_chain": "$\\tfrac{1}{2}(6)(4) = 12$.\n\n$\\boxed{12}$"},
    {"question": "Simplify $\\dfrac{3}{4} + \\dfrac{1}{6}$.",
     "optimized_chain": "LCD = 12: $\\dfrac{9}{12} + \\dfrac{2}{12} = \\dfrac{11}{12}$.\n\n$\\boxed{\\dfrac{11}{12}}$"},
    {"question": "What is the value of $x$ if $2x + 5 = 17$?",
     "optimized_chain": "$2x = 17 - 5 = 12 \\Rightarrow x = 6$.\n\n$\\boxed{6}$"},
]

_EXAMPLES = {
    "gsm8k":  {"standard_cot": _GSM_STD, "fast_solve": _GSM_FAST,
               "reduction": _GSM_RED, "cod": _GSM_COD, "ours_icl": _GSM_PNS},
    "math500": {"standard_cot": _MATH_STD, "fast_solve": _MATH_FAST,
                "reduction": _MATH_RED, "cod": _MATH_COD, "ours_icl": _MATH_PNS},
}

_SYSTEM = {
    "standard_cot": "You are a helpful assistant. Solve problems step by step, showing every reasoning step clearly.",
    "fast_solve":   "You are a helpful assistant. Solve problems efficiently. Provide a brief but complete solution — skip obvious sub-steps.",
    "reduction":    "You are a helpful assistant. Solve problems concisely. Include necessary logical steps but omit trivially redundant ones.",
    "cod":          "You are a helpful assistant. Solve problems using only the minimum key draft steps. Be extremely concise.",
    "ours_icl":     "You are a helpful assistant. Solve problems with sufficient and necessary reasoning: include every step needed for correctness; omit any step that is redundant.",
}

METHOD_LABELS = {
    "standard_cot": "Standard CoT",
    "fast_solve":   "Fast Solve",
    "reduction":    "Reduction",
    "cod":          "CoD",
    "ours_icl":     "Ours ICL",
}


def _fmt_shot(q, a):
    return f"Question: {q}\n\nSolution: {a}"


def build_messages(question: str, method: str, dataset: str) -> list:
    """Build chat messages for a given ICL method."""
    dt = "gsm8k" if "gsm" in dataset.lower() else "math500"
    system = _SYSTEM[method]
    examples = _EXAMPLES[dt][method]

    if method == "ours_icl":
        header = "### Examples — sufficient and necessary reasoning (include only causally essential steps):\n\n"
        shots  = [_fmt_shot(e["question"], e["optimized_chain"]) for e in examples]
    else:
        headers = {
            "standard_cot": "### Examples — solve step by step:\n\n",
            "fast_solve":   "### Examples — solve briefly:\n\n",
            "reduction":    "### Examples — solve concisely:\n\n",
            "cod":          "### Examples — key draft steps only:\n\n",
        }
        header = headers[method]
        shots  = [_fmt_shot(e["q"], e["a"]) for e in examples]

    user_content = header + "\n\n---\n\n".join(shots) + "\n\n---\n\n" + \
                   f"Question: {question}\n\nSolution:"
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_content},
    ]


# ── Verify data consistency ───────────────────────────────────────────────────
def _verify_consistency():
    gsm_q  = {m: [e["q"] if "q" in e else e["question"] for e in _EXAMPLES["gsm8k"][m]]
               for m in ["standard_cot","fast_solve","reduction","cod","ours_icl"]}
    math_q = {m: [e["q"] if "q" in e else e["question"] for e in _EXAMPLES["math500"][m]]
               for m in ["standard_cot","fast_solve","reduction","cod","ours_icl"]}

    gsm_ok  = all(gsm_q[m]  == gsm_q["standard_cot"]  for m in gsm_q)
    math_ok = all(math_q[m] == math_q["standard_cot"] for m in math_q)

    print("Few-shot example consistency check:")
    print(f"  GSM8K   base questions identical across all 5 methods: {'✓ PASS' if gsm_ok  else '✗ FAIL'}")
    print(f"  MATH-500 base questions identical across all 5 methods: {'✓ PASS' if math_ok else '✗ FAIL'}")
    print()
    print("GSM8K few-shot questions:")
    for q in gsm_q["standard_cot"]: print(f"  • {q[:70]}")
    print("MATH-500 few-shot questions:")
    for q in math_q["standard_cot"]: print(f"  • {q[:70]}")

_verify_consistency()
print("\nICL prompt builders loaded — methods:", list(METHOD_LABELS.keys()))

## 6 · Generation helper

Qwen3 is run with `enable_thinking=False` for ICL — standard instruction-following mode, no extended thinking.

In [ ]:
@torch.no_grad()
def generate_icl(messages: list, dataset: str) -> dict:
    """Run one ICL query through the base model and return response + metrics."""
    try:
        prompt = tokenizer.apply_chat_template(
            messages, enable_thinking=False, tokenize=False, add_generation_prompt=True
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    in_len = enc["input_ids"].shape[1]

    max_new = MAX_NEW_TOKENS.get(dataset, 1536)   # 1536 for gsm8k, 8192 for math500

    t0 = time.time()
    out = model.generate(
        **enc,
        max_new_tokens=max_new,
        do_sample=False,
        temperature=1.0,
        top_p=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - t0

    new_ids  = out[0][in_len:]
    response = tokenizer.decode(new_ids, skip_special_tokens=True)
    response = strip_think(response)

    extracted = extract_answer(response, dataset)
    return {
        "response":   response,
        "extracted":  extracted,
        "new_tokens": int(len(new_ids)),
        "steps":      count_steps(response),
        "seconds":    round(elapsed, 1),
    }


print("Generation helper ready.")

## 7 · Test data (10 questions)

Same 10 questions used for the self-distillation inference notebook: 8 GSM8K + 2 MATH-500.

In [ ]:
TEST_SAMPLES = [
    # ---- GSM8K — simple (warm-up) ----
    {"dataset": "gsm8k", "answer": "18",
     "question": "Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?"},
    {"dataset": "gsm8k", "answer": "540",
     "question": "James decides to run 3 sprints 3 times a week. He runs 60 meters each sprint. How many total meters does he run a week?"},
    # ---- GSM8K — multi-step ----
    {"dataset": "gsm8k", "answer": "10",
     "question": "Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?"},
    {"dataset": "gsm8k", "answer": "5",
     "question": "Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?"},
    {"dataset": "gsm8k", "answer": "42",
     "question": "Julie is reading a 120-page book. Yesterday, she was able to read 12 pages and today, she read twice as many pages as yesterday. If she wants to read half of the remaining pages tomorrow, how many pages should she read tomorrow?"},
    {"dataset": "gsm8k", "answer": "35",
     "question": "Mark has a garden with flowers. He planted plants of three different colors in it. Ten of them are yellow, and there are 80% more of those in purple. There are only 25% as many green flowers as there are yellow and purple flowers. How many flowers does Mark have in his garden?"},
    {"dataset": "gsm8k", "answer": "64",
     "question": "Kylar went to the store to buy glasses for his new apartment. One glass costs $5, but every second glass costs only 60% of the price. Kylar wants to buy 16 glasses. How much does he need to pay for them?"},
    {"dataset": "gsm8k", "answer": "25",
     "question": "Sam bought a dozen boxes, each with 30 highlighter pens inside, for $10 each box. He rearranged five of these boxes into packages of six highlighters each and sold them for $3 per package. He sold the rest of the highlighters separately at the rate of three pens per dollar. How much profit did he make in total, in dollars?"},
    # ---- MATH-500 ----
    {"dataset": "math500", "answer": "\\left( 3, \\frac{\\pi}{2} \\right)",
     "question": "Convert the point $(0,3)$ in rectangular coordinates to polar coordinates. Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$"},
    {"dataset": "math500", "answer": "9",
     "question": "How many positive whole-number divisors does 196 have?"},
]

print(f"Test set: {len(TEST_SAMPLES)} problems")
gsm  = sum(1 for s in TEST_SAMPLES if s["dataset"] == "gsm8k")
math = sum(1 for s in TEST_SAMPLES if s["dataset"] == "math500")
print(f"  GSM8K: {gsm}  |  MATH-500: {math}")
for i, s in enumerate(TEST_SAMPLES, 1):
    print(f"  [{i:02d}] {s['dataset']:7s}  ans={s['answer'][:20]!r:22s}  {s['question'][:55]}")

## 8 · Run all 5 ICL methods\n\nRuns each method on all 10 test problems (50 calls total). Expect **~10–15 minutes on T4**.\n\nProgress is printed after every question so you can monitor accuracy in real time.

In [ ]:
METHODS = ["standard_cot", "fast_solve", "reduction", "cod", "ours_icl"]

# all_results[method][i] = {response, extracted, new_tokens, steps, seconds}
all_results = {m: [] for m in METHODS}

for method in METHODS:
    label = METHOD_LABELS[method]
    print(f"\n{'='*65}")
    print(f"  METHOD: {label}")
    print(f"{'='*65}")
    correct = 0
    for i, sample in enumerate(TEST_SAMPLES, 1):
        msgs = build_messages(sample["question"], method, sample["dataset"])
        r    = generate_icl(msgs, sample["dataset"])
        ok   = is_correct(r["extracted"], sample["answer"])
        correct += int(ok)
        all_results[method].append({**r, "correct": ok, **sample})
        mark = "[OK]" if ok else "[X] "
        print(f"  {mark} #{i:02d} {sample['dataset']:7s} "
              f"expected={sample['answer'][:15]!r:17s} "
              f"got={r['extracted'][:15]!r:17s} "
              f"tok={r['new_tokens']:4d}  {r['seconds']}s")
    print(f"  → {label}: {correct}/{len(TEST_SAMPLES)} correct  "
          f"({correct/len(TEST_SAMPLES)*100:.0f}%)")

print("\nAll methods done.")

## 9 · Results table (paper Table 2 format)

Accuracy, average tokens, and average reasoning steps per method and dataset.

In [ ]:
gsm_idx  = [i for i, s in enumerate(TEST_SAMPLES) if s["dataset"] == "gsm8k"]
math_idx = [i for i, s in enumerate(TEST_SAMPLES) if s["dataset"] == "math500"]

print(f"\nModel: {BASE_MODEL}  |  ICL (no fine-tuning)  |  n={len(TEST_SAMPLES)} test problems")
print(f"{'─'*95}")
hdr = f"{'Method':<18} {'GSM8K Acc':>10} {'MATH Acc':>10} {'Overall Acc':>12} {'Avg Tokens':>11} {'Avg Steps':>10}"
print(hdr)
print(f"{'─'*95}")

summary_rows = []
for method in METHODS:
    res = all_results[method]
    label = METHOD_LABELS[method]

    gsm_correct  = sum(res[i]["correct"] for i in gsm_idx)
    math_correct = sum(res[i]["correct"] for i in math_idx)
    total_correct = gsm_correct + math_correct

    gsm_acc  = gsm_correct  / max(len(gsm_idx), 1)  * 100
    math_acc = math_correct / max(len(math_idx), 1) * 100
    total_acc = total_correct / len(TEST_SAMPLES) * 100

    avg_tok  = sum(r["new_tokens"] for r in res) / len(res)
    avg_step = sum(r["steps"]      for r in res) / len(res)

    marker = " ◄" if method == "ours_icl" else ""
    print(f"{label:<18} {gsm_acc:>9.1f}%  {math_acc:>9.1f}%  {total_acc:>11.1f}%  "
          f"{avg_tok:>10.0f}  {avg_step:>9.1f}{marker}")

    summary_rows.append({
        "method": label, "gsm8k_acc": round(gsm_acc, 1),
        "math_acc": round(math_acc, 1), "overall_acc": round(total_acc, 1),
        "avg_tokens": round(avg_tok, 1), "avg_steps": round(avg_step, 1),
        "gsm8k_correct": gsm_correct, "gsm8k_total": len(gsm_idx),
        "math_correct":  math_correct, "math_total":  len(math_idx),
    })

print(f"{'─'*95}")
print(f"{'◄ = Ours ICL (PNS-guided)':<50}")

# Save to JSONL for further analysis
with open("/content/icl_results_table.jsonl", "w", encoding="utf-8") as f:
    for row in summary_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print("\nSaved → /content/icl_results_table.jsonl")

## 10 · Per-question breakdown (adapter-style view)

In [ ]:
col_w = 10
q_w   = 40

hdr_cols = "".join(f"{METHOD_LABELS[m]:>{col_w}}" for m in METHODS)
print(f"{'#':>3}  {'Dataset':<8}  {'Expected':<12}  {hdr_cols}  {'Question':<{q_w}}")
print("-" * (3 + 2 + 8 + 2 + 12 + 2 + col_w * len(METHODS) + 2 + q_w))

for i, sample in enumerate(TEST_SAMPLES):
    results_row = ""
    for method in METHODS:
        r  = all_results[method][i]
        ok = "Y" if r["correct"] else "N"
        tok = r["new_tokens"]
        results_row += f"{ok}/{tok:>4}tok".rjust(col_w)
    q_short = sample["question"][:q_w - 3] + "..."
    exp = str(sample["answer"])[:12]
    print(f"{i+1:>3}  {sample['dataset']:<8}  {exp:<12}  {results_row}  {q_short}")

print("\n(Y=correct, N=wrong; tok=tokens generated)")

## 11 · Token efficiency comparison

The core claim: **Ours ICL** achieves competitive accuracy with fewer tokens than Standard CoT — by prompting the model with causally-necessary PNS-filtered exemplars.

In [ ]:
std_res  = all_results["standard_cot"]
ours_res = all_results["ours_icl"]

std_tok  = sum(r["new_tokens"] for r in std_res)  / len(std_res)
ours_tok = sum(r["new_tokens"] for r in ours_res) / len(ours_res)
std_acc  = sum(r["correct"] for r in std_res)  / len(std_res)  * 100
ours_acc = sum(r["correct"] for r in ours_res) / len(ours_res) * 100

reduction = (std_tok - ours_tok) / std_tok * 100
acc_delta = ours_acc - std_acc

print("Token efficiency — Ours ICL vs Standard CoT")
print(f"  Standard CoT : acc={std_acc:.1f}%  avg_tokens={std_tok:.0f}")
print(f"  Ours ICL     : acc={ours_acc:.1f}%  avg_tokens={ours_tok:.0f}")
print(f"  Delta        : acc={acc_delta:+.1f} pp  token_reduction={reduction:.1f}%")
print()
# Per method token counts
print(f"{'Method':<18}  {'Avg Tokens':>11}  {'vs Standard CoT':>16}")
print("-" * 50)
for method in METHODS:
    avg = sum(r["new_tokens"] for r in all_results[method]) / len(all_results[method])
    delta = avg - std_tok
    marker = " ◄" if method == "ours_icl" else ""
    print(f"{METHOD_LABELS[method]:<18}  {avg:>11.0f}  {delta:>+15.0f}{marker}")
print("\n◄ = Ours ICL (PNS-guided)")

## 12 · Download results

Download the JSONL results table to your local machine.

In [ ]:
from google.colab import files
files.download("/content/icl_results_table.jsonl")